# 🎄 Santa's Tree Packing Challenge 2025 🎅

## Shake & Optimize: A Simple Yet Effective Approach

**Competition:** [Santa 2025 - Christmas Tree Packing](https://www.kaggle.com/competitions/santa-2025)

**Approach:** The idea is simple - shake the solution in almost random directions and wait to see if the size decreases. This straightforward strategy leverages local search with random perturbations to escape local minima and find better packing configurations.

*Note: Code may be published after the competition ends.*

---

In [ ]:
source_file = "/kaggle/input/new-simple-fix-rebuild-large-layout-general/submission.csv"

In [ ]:
!santa-2025-shaker/shake_public --input="$source_file" --output="submission.csv"

In [ ]:
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from shapely.strtree import STRtree

# Set precision for Decimal (25 is good for contest standards)
getcontext().prec = 25
scale_factor = Decimal("1e18")


class ChristmasTree:
    """Represents a single, rotatable Christmas tree of a fixed size."""
    
    def __init__(self, center_x="0", center_y="0", angle="0"):
        """Initializes the Christmas tree with a specific position and rotation."""
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)
        
        # Tree dimensions
        trunk_w = Decimal("0.15")
        trunk_h = Decimal("0.2")
        base_w = Decimal("0.7")
        mid_w = Decimal("0.4")
        top_w = Decimal("0.25")
        tip_y = Decimal("0.8")
        tier_1_y = Decimal("0.5")
        tier_2_y = Decimal("0.25")
        base_y = Decimal("0.0")
        trunk_bottom_y = -trunk_h
        
        # Define the 15 vertices of the tree polygon
        initial_polygon = Polygon([
            (Decimal("0.0") * scale_factor, tip_y * scale_factor),
            (top_w / Decimal("2") * scale_factor, tier_1_y * scale_factor),
            (top_w / Decimal("4") * scale_factor, tier_1_y * scale_factor),
            (mid_w / Decimal("2") * scale_factor, tier_2_y * scale_factor),
            (mid_w / Decimal("4") * scale_factor, tier_2_y * scale_factor),
            (base_w / Decimal("2") * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal("2") * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal("2") * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal("2")) * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal("2")) * scale_factor, base_y * scale_factor),
            (-(base_w / Decimal("2")) * scale_factor, base_y * scale_factor),
            (-(mid_w / Decimal("4")) * scale_factor, tier_2_y * scale_factor),
            (-(mid_w / Decimal("2")) * scale_factor, tier_2_y * scale_factor),
            (-(top_w / Decimal("4")) * scale_factor, tier_1_y * scale_factor),
            (-(top_w / Decimal("2")) * scale_factor, tier_1_y * scale_factor),
        ])
        
        # Apply rotation and translation
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor)
        )


def has_overlap(trees):
    """Check if any two ChristmasTree polygons overlap."""
    if len(trees) <= 1:
        return False
    
    polygons = [t.polygon for t in trees]
    tree_index = STRtree(polygons)
    
    for i, poly in enumerate(polygons):
        indices = tree_index.query(poly)
        for idx in indices:
            if idx == i:
                continue
            if poly.intersects(polygons[idx]) and not poly.touches(polygons[idx]):
                return True
    return False


def load_trees_for_n(n, df):
    """Load all trees for a given N from the submission DataFrame."""
    group_data = df[df["id"].str.startswith(f"{n:03d}_")]
    trees = []
    for _, row in group_data.iterrows():
        x = str(row["x"]).lstrip('s')
        y = str(row["y"]).lstrip('s')
        deg = str(row["deg"]).lstrip('s')
        if x and y and deg:
            trees.append(ChristmasTree(x, y, deg))
    return trees


def fix_invalid_submission(new_csv_path, valid_csv_path, output_csv_path, max_n=200):
    """
    Fix invalid configurations in a submission by replacing them with valid ones.
    
    Args:
        new_csv_path: Path to the new submission CSV (may contain invalid configs)
        valid_csv_path: Path to the valid reference submission CSV
        output_csv_path: Path where the fixed submission will be saved
        max_n: Maximum N to check (default: 200)
    
    Returns:
        List of N values that were replaced
    """
    df_new = pd.read_csv(new_csv_path)
    df_valid = pd.read_csv(valid_csv_path)
    
    replaced_n = []
    
    for n in range(1, max_n + 1):
        trees = load_trees_for_n(n, df_new)
        if trees and has_overlap(trees):
            # Replace this group with valid configuration
            prefix = f"{n:03d}_"
            df_new = df_new[~df_new["id"].str.startswith(prefix)]
            df_replacement = df_valid[df_valid["id"].str.startswith(prefix)]
            df_new = pd.concat([df_new, df_replacement])
            replaced_n.append(n)
    
    # Sort and save
    df_new = df_new.sort_values(by="id").reset_index(drop=True)
    df_new.to_csv(output_csv_path, index=False)
    
    if replaced_n:
        print(f"Fixed {len(replaced_n)} invalid configurations: N={replaced_n}")
    else:
        print("No invalid configurations found. Submission is valid.")
    
    return replaced_n


_ = fix_invalid_submission(new_csv_path="submission.csv", valid_csv_path=source_file, output_csv_path="submission_fixed.csv")